# 統計学特講

## 04 物価指数

### 目的
- ラスパイレス価格指数を計算する
- インフレ率を計算する
- CPIとコアCPIを比較する
- 実質賃金を計算する

### 前準備

In [ ]:
import pandas as pd              # データフレーム関連のおまじない
import matplotlib.pyplot as plt  # グラフ描画関連のおまじない

### 4-1 ラスパイレス価格指数

#### ラスパイレス価格指数とは

ラスパイレス価格指数は，**基準年の購入量**を固定して，価格だけが変わったときに支出額がどれだけ変わるかを測る指数である．

今回は，日本の品目データ（授業用に若干加工済）を使って，基準年から比較年にかけての価格指数を計算する．

#### CSVデータをデータフレームに読み込み

In [ ]:
data_url = "https://raw.githubusercontent.com/tnarumi/lecture-statistics/main/data/laspeyres_price_items_japan.csv"
df_laspeyres_items = pd.read_csv(data_url)
df_laspeyres_items.columns

```item```: 品目，
```price_base```: 基準年の単位あたり価格，
```quantity_base```: 基準年の年間購入量，
```price_current```: 比較年の単位あたり価格，
```quantity_current```: 比較年の年間購入量

品目ごとに価格や数量の単位は異なる．価格は単位あたり価格，数量はその価格単位に対応する年間購入量として解釈する．

#### データの確認

##### データフレームの概要を確認

In [ ]:
df_laspeyres_items.info()

##### 内容の確認

In [ ]:
df_laspeyres_items

たとえば，食パンの価格は1kgあたり，数量はkg/年として解釈する．

##### 基本統計量の一覧表示

In [ ]:
df_laspeyres_items.describe()

##### ラスパイレス価格指数の式

$$
P_L(t)
=
\frac{\sum_i p_{ti}q_{0i}}
{\sum_i p_{0i}q_{0i}}
\times100
$$

- $p_{0i}$：基準年の品目 $i$ の価格
- $q_{0i}$：基準年の品目 $i$ の購入量
- $p_{ti}$：比較年の品目 $i$ の価格

ポイントは，分子でも比較年の購入量ではなく，**基準年の購入量**を使うことである．

#### 分母の計算

分母は，基準年価格と基準年購入量の積の合計である．

In [ ]:
denominator = (df_laspeyres_items["price_base"] * df_laspeyres_items["quantity_base"]).sum()
denominator

＊ 合計の積 ```df_laspeyres_items["price_base"].sum() * df_laspeyres_items["quantity_base"].sum()``` とは異なることに注意

#### 分子の計算

分子は，比較年価格と **基準年購入量** の積の合計である．

In [ ]:
numerator = (df_laspeyres_items["price_current"] * df_laspeyres_items["quantity_base"]).sum()
numerator

#### 指数の計算

In [ ]:
laspeyres = numerator / denominator * 100
laspeyres

指数は，基準年を100とするものなので，比較年の指数を $x$ とすると，基準年から比較年にかけて物価が約 $(x-100)$％変化したことになる．

#### 発展：支出割合（ウエイト）

支出額に占める各品目の割合を，ウエイトという．ラスパイレス価格指数では基準年のウエイト

$$
w_{0i}
=
\frac{p_{0i}q_{0i}}
{\sum_i p_{0i}q_{0i}}
$$

が重要．

In [ ]:
df_laspeyres_items["expenditure_base"] = df_laspeyres_items["price_base"] * df_laspeyres_items["quantity_base"]
df_laspeyres_items["weight_base"] = df_laspeyres_items["expenditure_base"] / df_laspeyres_items["expenditure_base"].sum()
df_laspeyres_items.nlargest(10, "weight_base")[["item", "weight_base"]]

#### ウエイトの合計を確認

支出割合なので，ウエイトの合計は理論的に 1 になる．

In [ ]:
df_laspeyres_items["weight_base"].sum()

#### ウエイトを用いた指数の計算

基準年ウエイト $w_{0i}$ を用いると，ラスパイレス価格指数は

$$
P_L(t) 
=
\frac{\sum_i p_{ti}q_{0i}} {\sum_i p_{0i}q_{0i}} \times100
=
\sum_i \frac{p_{ti}}{p_{0i}} \frac{p_{0i}q_{0i}}{\sum_i p_{0i}q_{0i}} \times 100
=
\sum_i w_{0i} \frac{p_{ti}}{p_{0i}}\times100
$$

とも表現できる．つまり，ラスパイレス指数は基準年ウエイトと価格比の積の合計との見方もできる．

In [ ]:
df_laspeyres_items["price_ratio"] = df_laspeyres_items["price_current"] / df_laspeyres_items["price_base"]
laspeyres = (df_laspeyres_items["weight_base"] * df_laspeyres_items["price_ratio"]).sum() * 100
laspeyres

#### 考察問題

1. 比較年数量を基準とする価格指数

$$
P_P(t) = 
\frac{\sum_i p_{ti}q_{ti}}
{\sum_i p_{0i}q_{ti}}
\times100
$$

をパーシェ価格指数という．
パーシェ価格指数を計算しなさい．
また，ラスパイレス価格指数との比較を通じて，それぞれの価格指数について考察しなさい．

2. 基準年の購入量ではなく比較年の購入量を使って

$$
\frac{\sum_i p_{ti}q_{ti}}
{\sum_i p_{0i}q_{0i}}
\times100
$$

として得られる値を求めなさい．
また，この量を価格指数と呼ぶことは適当かどうかを議論しなさい．

In [ ]:
# ここにコードを各自で追加してください



### 4-2 消費者物価指数（CPI）

#### CPIとは

CPI（Consumer Price Index，消費者物価指数）は，家計が購入する財・サービスの価格が，全体としてどれだけ変化したかを表す指数である．

#### CSVデータをデータフレームに読み込み

In [ ]:
data_price = "https://raw.githubusercontent.com/tnarumi/lecture-statistics/main/data/price_index_japan.csv"
df_price = pd.read_csv(data_price)

```date```：年月，
```CPI```：消費者物価指数（総合），
```Core_CPI```：コア消費者物価指数（生鮮食品を除く総合），
```CGPI```：企業物価指数，
```nominal_wage```：名目賃金指数

#### データフレームの概要を確認

In [ ]:
df_price.info()

#### 基本統計量の一覧表示

In [ ]:
df_price.describe()

#### 日付データに変換

CSVから読み込んだ年月を，Pythonで日付として扱える形に変換する．

In [ ]:
df_price["date"] = pd.to_datetime(df_price["date"])
df_price

#### CPIの時系列プロット

In [ ]:
df_price.plot(x="date", y="CPI", figsize=(10, 4), marker="o", grid=True)

plt.xlabel("date")
plt.ylabel("index")
plt.title("Consumer Price Index")
plt.show()

グラフを見ると，CPIの長期的な動きがわかる．

#### 考察問題

1. CPIは長期的に上昇しているか，それとも低下しているか．
2. 2022年以降の動きには，どのような特徴があるか．

### 4-3 インフレ率

#### インフレ率とは

インフレ率は，物価指数がどれだけ変化したかを表す．

月次データでは，前年の同じ月と比べる **前年同月比** がよく使われる．

$$
\pi_t
=
\frac{CPI_t-CPI_{t-12}}
{CPI_{t-12}}
\times100
$$

#### 12か月前のCPIを作る

`shift(12)` は，12行前の値を取り出す関数である．月次データなので，12行前は前年同月に対応する．

In [ ]:
df_price["CPI_lag12"] = df_price["CPI"].shift(12)
df_price[["date", "CPI", "CPI_lag12"]].head(15)

最初の12か月は，前年同月のデータがないため，インフレ率はNaNになる．NaNはデータが存在しないことを表す．

#### 定義式に沿ってインフレ率を計算

In [ ]:
df_price["inflation_rate"] = (
    (df_price["CPI"] - df_price["CPI_lag12"])
    / df_price["CPI_lag12"]
    * 100
)
df_price[["date", "CPI", "CPI_lag12", "inflation_rate"]].head(15)

#### pct_changeを使った計算

`pct_change(12)` は，12行前からの変化率を計算する関数である．定義式で計算した結果と同じになることを確認する．

In [ ]:
df_price["inflation_rate_pct_change"] = df_price["CPI"].pct_change(12) * 100
df_price[["date", "inflation_rate", "inflation_rate_pct_change"]].head(20)

#### インフレ率の時系列プロット

In [ ]:
df_price.plot(x="date", y="inflation_rate", figsize=(10, 4), marker="o", grid=True)

plt.axhline(0, color="black", linewidth=1)
plt.xlabel("date")
plt.ylabel("inflation rate (%)")
plt.title("Inflation rate")
plt.show()

インフレ率がプラスであれば，前年同月と比べて物価が上がっていることを意味する．

インフレ率がマイナスであれば，前年同月と比べて物価が下がっていることを意味する．

#### 考察問題

1. インフレ率が最も高い月を確認せよ．
2. インフレ率がマイナスの月がある場合，その月には何が起きていると考えられるか．

### 4-4 コアCPI

#### コアCPIとは

ニュースで「コアCPI」という言葉が出てくることがある．日本では通常，**生鮮食品を除く総合CPI**を指す．

- CPI：消費者物価指数の総合指数
- コアCPI：生鮮食品を除く総合指数

生鮮食品は天候などで短期的に変動しやすいため，それを除いたコアCPIが経済全体の基調的な物価変動を見たいときに利用される．

国によってコアCPIの定義が異なることがあるので注意．例えば，アメリカでは食品とエネルギーを除くCPIをcore CPIと呼ぶ．なお，生鮮食品及びエネルギーを除く総合を，日本ではコアコアCPIと呼ぶことがある．

#### CPIとコアCPIを同じグラフに描く

In [ ]:
df_price.plot(x="date", y=["CPI", "Core_CPI"], figsize=(10, 4), marker="o", grid=True)

plt.xlabel("date")
plt.ylabel("index")
plt.title("CPI and Core CPI")
plt.legend(loc="best")
plt.show()

CPIとコアCPIは似た動きをするが，完全に同じではない．

生鮮食品の価格が大きく動く時期には，CPIとコアCPIの差が大きくなることがある．

#### CPI，コアCPI，CGPIを比較する

CGPI（Corporate Goods Price Index，企業物価指数）は，企業間で取引される財の価格を表す指数である．消費者が購入する財・サービスの価格を表すCPIとは対象が異なる．

In [ ]:
df_price.plot(x="date", y=["CPI", "Core_CPI", "CGPI"], figsize=(10, 5), marker="o", grid=True)

plt.xlabel("date")
plt.ylabel("index")
plt.title("CPI, Core CPI, and CGPI")
plt.legend(loc="best")
plt.show()

CGPIはCPIよりも変動が大きく見えることがある．これは，企業間の取引価格の変化が，すぐに消費者物価へ反映されるとは限らないためである．

#### 考察問題

1. CPIとコアCPIの差が大きい時期はあるか．
2. CGPIとCPIの動き方には，どのような違いがあるか．

### 4-5 実質賃金

#### 名目値と実質値

名目値は，その時点の金額や指数をそのまま見た値である．

実質値は，物価変化の影響を取り除いた値である．

たとえば名目賃金が上がっていても，物価がそれ以上に上がっていれば，買えるものは増えない可能性がある．


#### 実質賃金の式

$$
\text{実質賃金}
=
\frac{\text{名目賃金}}
{\text{CPI}}
\times100
$$

ここでは，名目賃金指数をCPIで割ることで，物価上昇の影響を取り除く．

#### 実質賃金指数を計算

In [ ]:
df_price["real_wage"] = (
    df_price["nominal_wage"]
    / df_price["CPI"]
    * 100
)
df_price[["date", "nominal_wage", "CPI", "real_wage"]].head()

#### 名目賃金指数と実質賃金指数を同じグラフに描く

In [ ]:
df_price.plot(x="date", y=["nominal_wage", "real_wage"], figsize=(10, 4), marker="o", grid=True)

plt.xlabel("date")
plt.ylabel("index")
plt.title("Nominal wage and real wage")
plt.legend(loc="best")
plt.show()

名目賃金が増加していても，実質賃金が同じように増加するとは限らない．物価上昇が大きいと，名目賃金の上昇があっても生活の実感は改善しにくい．

#### 考察問題

1. 名目賃金と実質賃金の動きは同じか，それとも違うか．
2. 物価上昇が大きい時期に，実質賃金はどのように動いているか．

### 4-6 練習用

課題：下記内容を確認しなさい．

1. CPIのインフレ率が最も高い月を確認せよ
2. CPI，コアCPI，CGPIのうち，最も変動が大きいものはどれか考えよ
3. 実質賃金が最も低い月を確認せよ
4. 名目賃金が上がっていても実質賃金が上がらない理由を説明せよ

In [ ]:
# ここにコードを各自で追加してください

